Licensed under the Apache License, Version 2.0


# OWL-v2 Fine-Tuning and Inference Workflow

This notebook demonstrates the end-to-end workflow for fine-tuning OWL-v2 and performing inference.

In [ ]:
import json
import os
from typing import Any, Dict, Tuple, Union

from absl import logging
import datasets
from etils import epath
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
import torch
import transformers

from Uboreshaji_Modeli.common import box_utils
from Uboreshaji_Modeli.common import config
from Uboreshaji_Modeli.common import data
from Uboreshaji_Modeli.common import losses
from Uboreshaji_Modeli.common import matcher
from Uboreshaji_Modeli.common import trainer
from Uboreshaji_Modeli.engines import factory


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 0. [Optional] Convert a COCO JSON dataset for use
The trainer needs a huggingface dataset, if you don't already have one use the
following cell.

In [ ]:
def _load_coco_json(json_path: epath.Path) -> Dict[str, Any]:
  """Loads a COCO-style JSON annotation file using epath."""
  with json_path.open("r") as f:
    return json.load(f)


def _coco_to_hf_dict(
    image_dir: epath.Path, annotation_file: epath.Path
) -> Tuple[Dict[str, Any], datasets.Features]:
  """Converts COCO annotations to a dictionary for Hugging Face Dataset.

  Args:
    image_dir: EPath to the directory containing images.
    annotation_file: EPath to the COCO JSON annotation file.

  Returns:
    A tuple containing:
      - hf_data: A dictionary structured for `datasets.Dataset.from_dict`.
      - features: A `datasets.Features` object defining the dataset schema.
  """
  coco_data = _load_coco_json(annotation_file)

  if "categories" not in coco_data:
    logging.warning("No categories found in annotation file.")
    return {}, datasets.Features()

  if "images" not in coco_data:
    logging.warning("No images found in annotation file.")
    return {}, datasets.Features()

  if "annotations" not in coco_data:
    logging.warning("No annotations found in annotation file.")
    return {}, datasets.Features()

  id2label = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
  # Ensure background is included if not present, as it's often used in models.
  if 0 not in id2label:
    id2label[0] = "background"
  sorted_labels = [id2label[i] for i in sorted(id2label.keys())]

  # Group annotations by image_id
  img_id_to_anns = {}
  for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in img_id_to_anns:
      img_id_to_anns[img_id] = []
    img_id_to_anns[img_id].append(ann)

  hf_data = {
      "image_id": [],
      "image": [],
      "objects": [],
  }

  for image_info in coco_data["images"]:
    image_id = image_info["id"]
    file_name = image_info["file_name"]
    image_path = image_dir / file_name

    if not image_path.exists():
      logging.warning("Warning: Image not found: %s", image_path)
      continue

    try:
      with image_path.open("rb") as f:
        image = Image.open(f).convert("RGB")
    except (IOError, OSError, Image.UnidentifiedImageError) as e:
      logging.warning("Warning: Could not open image %s: %s", image_path, e)
      continue

    annotations = img_id_to_anns.get(image_id, [])
    categories = []
    bboxes = []
    for ann in annotations:
      # COCO bbox format: [xmin, ymin, width, height]
      bbox = ann["bbox"]
      if isinstance(bbox, dict):
        bbox = [
            float(bbox["x"]),
            float(bbox["y"]),
            float(bbox["width"]),
            float(bbox["height"]),
        ]
      category_id = ann["category_id"]
      categories.append(category_id)
      bboxes.append(bbox)

    hf_data["image_id"].append(image_id)
    hf_data["image"].append(image)
    hf_data["objects"].append({"category": categories, "bbox": bboxes})

  features = datasets.Features(
      image_id=datasets.Value(dtype="int64"),
      image=datasets.Image(),
      objects=datasets.Sequence(
          feature={
              "category": datasets.ClassLabel(names=sorted_labels),
              "bbox": datasets.Sequence(
                  feature=datasets.Value(dtype="float32"), length=4
              ),
          }
      ),
  )

  return hf_data, features


def convert_coco_folder_to_hf(
    coco_root_dir: Union[str, epath.Path],
    output_hf_path: Union[str, epath.Path]) -> None:
  """Converts a COCO-formatted folder to a Hugging Face DatasetDict.

  This function is adapted to the user's specific COCO directory structure:
  - Annotation files: train.json, valid.json, test.json are at the root.
  - All images are in a single 'images/' folder at the root.

  Args:
    coco_root_dir: Path to the root directory containing COCO files.
    output_hf_path: Path where the Hugging Face DatasetDict will be saved.
  """
  coco_root_dir = epath.Path(coco_root_dir)
  output_hf_path = epath.Path(output_hf_path)
  dataset_dict = {}
  image_dir = coco_root_dir / "images"

  if not image_dir.exists():
    logging.error("Error: Images directory not found at %s", image_dir)
    return

  split_configs = {
      "train": coco_root_dir / "train.json",
      "valid": coco_root_dir / "valid.json",
      "test": coco_root_dir / "test.json",
  }

  for split, annotation_file in split_configs.items():
    if not annotation_file.exists():
      logging.warning(
          "Skipping split '%s': Annotation file not found at %s",
          split,
          annotation_file,
      )
      continue

    logging.info("Processing split: %s...", split)
    logging.info("  - Annotations: %s", annotation_file)
    logging.info("  - Images: %s", image_dir)

    hf_data, features = _coco_to_hf_dict(image_dir, annotation_file)
    if not hf_data["image_id"]:
      logging.warning("Skipping split '%s': No valid images found.", split)
      continue

    dataset = datasets.Dataset.from_dict(hf_data, features=features)
    dataset_dict[split] = dataset
    logging.info("  - Loaded %d samples for '%s'.", len(dataset), split)

  if not dataset_dict:
    logging.warning("No valid COCO splits found to convert.")
    return

  hf_dataset_dict = datasets.DatasetDict(dataset_dict)
  hf_dataset_dict.save_to_disk(output_hf_path)
  logging.info("Dataset saved to %s", output_hf_path)

_COCO_ROOT_DIR = "/path/to/my/coco_root"  # @param {type: "string"}
_OUTPUT_HF_PATH = "/path/to/my/coco_root"  # @param {type: "string"}
convert_coco_folder_to_hf(
      _COCO_ROOT_DIR,
      _OUTPUT_HF_PATH,
  )


## 1. Create and Customize Configuration
We initialize the base configuration and override specific fields for our current maize dataset experiment.

In [ ]:
cfg = config.get_base_config()

# Customize for Maize dataset experiment
cfg.experiment_name = "maize_owlv2_finetune_nb"
cfg.dataset.dataset_path = (
    "/path/to/huggingface_dataset/"
)
cfg.training.batch_size = 2
cfg.training.num_train_epochs = 1

print(f"Experiment: {cfg.experiment_name}")

## 2. Prepare Data
Load the dataset and prepare the training split with necessary transformations.

In [ ]:
engine = factory.get_engine(cfg.model_flavor)
model, processor = engine.load_model_and_processor(cfg.model_id, device)

dataset_root = data.get_dataset(cfg)
train_split = dataset_root[cfg.dataset.train_split]

# Prepare dataset with transforms
dataset_id2label = train_split.features["objects"]["category"].feature.names
valid_categories = [
    n for n in dataset_id2label if n not in cfg.dataset.exclude_classes
]
model_label2id = {name: i for i, name in enumerate(valid_categories)}

transform_fn = engine.get_transform_fn(
    processor, valid_categories, dataset_id2label, model_label2id
)
train_dataset = train_split.with_transform(transform_fn)
eval_dataset = dataset_root[cfg.dataset.eval_split].with_transform(transform_fn)

## 3. Initialize Model and Trainer
Set up the OWL-v2 model, the Hungarian Matcher, Criterion, and the Custom Trainer.

In [ ]:
criterion, weight_dict = engine.get_criterion(len(valid_categories), cfg, device)

training_args = transformers.TrainingArguments(
    output_dir="/tmp/notebook_train",
    per_device_train_batch_size=cfg.training.batch_size,
    num_train_epochs=cfg.training.num_train_epochs,
    remove_unused_columns=False,
    report_to="none",
)

custom_trainer = trainer.CustomTrainer(
    model=model,
    args=training_args,
    data_collator=engine.get_collate_fn(cfg),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    criterion=criterion,
    weight_dict=weight_dict,
)

## 4. Fine-Tuning (Example Run)
OPTIONAL: Uncomment the line below to start a short training session.

In [ ]:
# custom_trainer.train()

## 5. Inference and Visualization
Run inference on a test sample and visualize the predicted bounding boxes.

In [ ]:
def run_inference(image, processor, model, text, threshold=0.1):
  width, height = image.size
  inputs = processor(text=text, images=image, return_tensors="pt").to(device)
  with torch.no_grad():
    outputs = model(**inputs)

  target_sizes = torch.tensor([[height, width]])
  results = processor.post_process_object_detection(
      outputs, threshold=threshold, target_sizes=target_sizes
  )
  return results[0]


def visualize(image, results, categories):
  draw = ImageDraw.Draw(image)
  for score, label, box in zip(
      results["scores"], results["labels"], results["boxes"]
  ):
    box = [round(i, 2) for i in box.tolist()]
    draw.rectangle(box, outline="red", width=3)
    draw.text((box[0], box[1]), f"{categories[label]}: {score:.2f}", fill="red")

  plt.figure(figsize=(10, 10))
  plt.imshow(image)
  plt.axis("off")
  plt.show()


# Run on a test image
sample = dataset_root["test"][0]
test_image = sample["image"]
res = run_inference(
    test_image, processor, model, [text_inputs[0]]
)  # Example with one category
visualize(test_image, res, text_inputs)